# ML Mini-Project: Reproduce and Extend
**Reference:** *Predicting Eruptive Events at Volcanoes from Earthquake Data* (Mullet, Singh, Wong; Stanford CS229, 2019).

- **Part 1:** reproduce the paper's pipeline (logistic regression, K-means, random forest, neural network) on the Kilauea earthquake catalog and compare with the reported results.
- **Part 2:** add one model not used in the paper (RBF support vector machine) and evaluate it on the same split and metrics.

## Part 1: Reproduction

In [1]:
!pip install -q requests pandas numpy scikit-learn torch matplotlib
import numpy as np, pandas as pd, requests, io, torch, torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import cohen_kappa_score, roc_auc_score, confusion_matrix, mean_squared_error, r2_score
SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)

In [2]:
def get_quakes(start_year, end_year, minmag=0):
    frames = []
    for y in range(start_year, end_year + 1):
        r = requests.get("https://earthquake.usgs.gov/fdsnws/event/1/query", params=dict(
            format="csv", starttime=f"{y}-01-01", endtime=f"{y}-12-31",
            minlatitude=19.1, maxlatitude=19.5, minlongitude=-155.5, maxlongitude=-155.0,
            minmagnitude=minmag, orderby="time-asc", limit=20000))
        frames.append(pd.read_csv(io.StringIO(r.text)))
    df = pd.concat(frames)
    df["time"] = pd.to_datetime(df["time"]).dt.tz_localize(None)
    return df[["time","longitude","latitude","depth","mag"]].dropna().sort_values("time").reset_index(drop=True)

# Download everything once, then filter locally
eq_all = get_quakes(2000, 2016, minmag=0)
print("all events:", len(eq_all))

TARGET = 3764
cutoffs = [0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0]
for mm in cutoffs:
    print(f"minmag {mm}: {(eq_all.mag >= mm).sum()}")

MINMAG = min(cutoffs, key=lambda mm: abs((eq_all.mag >= mm).sum() - TARGET))
eq = eq_all[eq_all.mag >= MINMAG].reset_index(drop=True)
print(f"\nchosen MINMAG = {MINMAG}, N = {len(eq)}")

all events: 36192
minmag 0.0: 36192
minmag 0.5: 35703
minmag 1.0: 32846
minmag 1.5: 20011
minmag 2.0: 6207
minmag 2.5: 1259
minmag 3.0: 285

chosen MINMAG = 2.0, N = 6207


## Data and Setup Choices

- **Earthquake catalog:** ANSS catalog via the USGS ComCat API (the paper's catalog, accessed through WOVOdat). Spatial box: longitude -155.5 to -155.0, latitude 19.1 to 19.5 (East Rift Zone near Pu'u 'O'o, read from the paper's Figure 1).
- **Date range and magnitude cutoff:** 2000-2016, minimum magnitude 2.0. The paper does not state these, so I chose the cutoff that came closest to the paper's 3,764 earthquakes on a coarse grid. This gave 6,207 events, or 5,950 after labeling (quakes after the last eruption start have no "next eruption" and are dropped). The catalog is therefore larger than the paper's.
- **Eruption labels:** Episodes used: 2007-06-17 to 2007-06-19, 2011-03-05 to 2011-03-09, and 2014-06-27 to 2016-05-24, taken from HVO / Smithsonian GVP reports. The paper's own eruption table is not published, so these are my approximation. Each earthquake is labeled erupting (1) or repose (0), and time-to-eruption is the days until the next eruption start (0 if erupting). About 18.5% of earthquakes are labeled erupting, close to the paper's roughly 20% class imbalance.
- **Features:** longitude, latitude, depth, magnitude, and earthquake counts over the previous 1, 7 and 30 days (computed within the filtered catalog).
- **Split and scaling:** stratified 70/20/10 train/dev/test split with seed 42. Features and the regression target were standardized using training-set statistics only.

In [3]:
# Eruption episodes (start, end), approximated from HVO / Smithsonian GVP reports.
# The paper does not publish its own eruption table (see "Data and Setup Choices").
episodes = [
    ("2007-06-17", "2007-06-19"),   # Father's Day intrusion/eruption, East Rift Zone
    ("2011-03-05", "2011-03-09"),   # Kamoamoa fissure eruption
    ("2014-06-27", "2016-05-24"),   # June 27 lava flow episode (end date approximate)
]
eruptions = pd.DataFrame({
    "start": pd.to_datetime([s for s, e in episodes]),
    "end":   pd.to_datetime([e for s, e in episodes]),
})

def label(t):
    inside = ((eruptions.start <= t) & (t <= eruptions.end)).any()
    if inside: return 1, 0.0
    nxt = eruptions.start[eruptions.start > t]
    return 0, ((nxt.min() - t).days if len(nxt) else np.nan)

lab = eq["time"].apply(label)
eq["erupting"] = [l[0] for l in lab]
eq["tte"] = [l[1] for l in lab]
eq = eq.dropna().reset_index(drop=True)   # drops quakes after the last eruption start

print("N after labeling:", len(eq))
print("erupting share:", round(eq.erupting.mean(), 3), "(target ~0.20)")
print(eq.groupby(eq.time.dt.year).erupting.agg(["size", "mean"]))   # sanity check per year

N after labeling: 5950
erupting share: 0.185 (target ~0.20)
      size      mean
time                
2000   526  0.000000
2001   175  0.000000
2002   233  0.000000
2003   396  0.000000
2004   310  0.000000
2005   288  0.000000
2006   399  0.000000
2007   534  0.299625
2008   297  0.000000
2009   207  0.000000
2010   141  0.000000
2011   361  0.144044
2012   541  0.000000
2013   394  0.000000
2014   398  0.354271
2015   602  1.000000
2016   148  1.000000


In [4]:
t = eq["time"].values.astype("datetime64[s]").astype(np.int64)
for name, days in [("rate1",1),("rate7",7),("rate30",30)]:
    eq[name] = np.arange(len(t)) - np.searchsorted(t, t - days*86400, side="left")

FEATS = ["longitude","latitude","depth","mag","rate1","rate7","rate30"]

In [5]:
X, yc, yr = eq[FEATS].values, eq["erupting"].values, eq["tte"].values
idx = np.arange(len(eq))
tr, tmp = train_test_split(idx, test_size=0.30, stratify=yc, random_state=SEED)
dv, te = train_test_split(tmp, test_size=1/3, stratify=yc[tmp], random_state=SEED)  # 20% / 10%

sx = StandardScaler().fit(X[tr]); sy = StandardScaler().fit(yr[tr].reshape(-1,1))
Xs = sx.transform(X); yrs = sy.transform(yr.reshape(-1,1)).ravel()
splits = {"train":tr, "dev":dv, "test":te}

In [6]:
def clf_metrics(y, score, thr):
    pred = (score >= thr).astype(int)
    return dict(kappa=cohen_kappa_score(y, pred), auroc=roc_auc_score(y, score), cm=confusion_matrix(y, pred).tolist())

def reg_metrics(y, p):
    return dict(rmse=mean_squared_error(y, p)**0.5, r2=r2_score(y, p))

def best_thr(y, score):
    ths = np.unique(np.quantile(score, np.linspace(0.05, 0.95, 91)))
    return max(ths, key=lambda t: cohen_kappa_score(y, (score >= t).astype(int)))

results = {}
def eval_clf(name, score_fn, thr=None):
    # threshold chosen on dev to maximize kappa, then applied to train/dev/test
    t = best_thr(yc[dv], score_fn(Xs[dv])) if thr is None else thr
    for s, i in splits.items():
        results[(name, "clf", s)] = clf_metrics(yc[i], score_fn(Xs[i]), t)

def eval_reg(name, pred_fn):
    for s, i in splits.items():
        results[(name, "reg", s)] = reg_metrics(yrs[i], pred_fn(Xs[i]))

In [7]:
# Logistic regression
lr = LogisticRegression(max_iter=1000).fit(Xs[tr], yc[tr])
eval_clf("LogReg", lambda X: lr.predict_proba(X)[:,1])

# K-means classification: 8 clusters per class
km_e = KMeans(8, n_init=10, random_state=SEED).fit(Xs[tr][yc[tr]==1])
km_r = KMeans(8, n_init=10, random_state=SEED).fit(Xs[tr][yc[tr]==0])
def km_score(X):  # higher means closer to the erupting clusters
    return km_r.transform(X).min(1) - km_e.transform(X).min(1)
eval_clf("KMeans", km_score)   # threshold tuned on dev, like the other classifiers

# K-means regression: k=10 on features plus target, match on features only
km_reg = KMeans(10, n_init=10, random_state=SEED).fit(np.c_[Xs[tr], yrs[tr]])
cent = km_reg.cluster_centers_
def km_reg_pred(X):
    d = ((X[:,None,:] - cent[None,:,:-1])**2).sum(-1)
    return cent[d.argmin(1), -1]
eval_reg("KMeans", km_reg_pred)

# Random forests: tune depth and number of trees on dev
best = (-1, None)
for d in [5,10,15,20,30]:
    for n in [100,300]:
        m = RandomForestClassifier(n_estimators=n, max_depth=d, max_features=3, random_state=SEED, n_jobs=-1).fit(Xs[tr], yc[tr])
        a = roc_auc_score(yc[dv], m.predict_proba(Xs[dv])[:,1])
        if a > best[0]: best = (a, m)
rf_c = best[1]
eval_clf("RF", lambda X: rf_c.predict_proba(X)[:,1])

best = (-9, None)
for d in [5,10,15,20,30]:
    for n in [100,300]:
        m = RandomForestRegressor(n_estimators=n, max_depth=d, max_features=3, random_state=SEED, n_jobs=-1).fit(Xs[tr], yrs[tr])
        a = r2_score(yrs[dv], m.predict(Xs[dv]))
        if a > best[0]: best = (a, m)
rf_r = best[1]
eval_reg("RF", rf_r.predict)

## Implementation Details and Deviations

- **Classification threshold:** The paper does not state its decision threshold. Because about 80% of labels are negative, the default 0.5 threshold gives near-zero kappa. I chose the threshold that maximizes Cohen's kappa on the development set, then applied it to train and test.
- **Random forest:** max_features = 3 as in the paper. The number of trees and maximum depth were tuned on the dev set (AUROC for classification, R² for regression) over depth {5, 10, 15, 20, 30} and trees {100, 300}, because the paper gives only the depth plateau (about 15).
- **Neural network:** 4 hidden layers, ReLU activations, SGD (momentum 0.9), batch size 50, weight decay, BCE loss for classification and MSE loss for regression. The paper does not report learning rate, weight decay, or number of epochs. I used four hidden layers of 1000 units, tuned learning rate {1e-3, 1e-2} and weight decay {1e-4, 1e-3} on the dev set (AUROC for classification, R² for regression), and trained for 100 epochs.
- **K-means:** k = 8 per class for classification (separate clusters for erupting and repose earthquakes), and k = 10 for regression with prediction by nearest centroid on features only, as described in the paper.

In [8]:
import torch, torch.nn as nn

dev = "cuda" if torch.cuda.is_available() else "cpu"

def make_net():
    L, d = [], Xs.shape[1]
    for _ in range(4):
        L += [nn.Linear(d, 1000), nn.ReLU()]; d = 1000
    return nn.Sequential(*L, nn.Linear(1000, 1)).to(dev)

def train_nn(task, lr=1e-3, wd=1e-4, epochs=100):
    torch.manual_seed(SEED)
    net = make_net()
    opt = torch.optim.SGD(net.parameters(), lr=lr, weight_decay=wd, momentum=0.9)
    lossf = nn.BCEWithLogitsLoss() if task == "clf" else nn.MSELoss()
    y = yc if task == "clf" else yrs
    Xt = torch.tensor(Xs[tr], dtype=torch.float32, device=dev)
    yt = torch.tensor(y[tr], dtype=torch.float32, device=dev)
    for ep in range(epochs):
        perm = torch.randperm(len(Xt), device=dev)
        for b in range(0, len(Xt), 50):
            j = perm[b:b+50]
            opt.zero_grad()
            lossf(net(Xt[j]).squeeze(1), yt[j]).backward()
            opt.step()
    return net

def nn_pred(net, task):
    def f(X):
        with torch.no_grad():
            o = net(torch.tensor(X, dtype=torch.float32, device=dev)).squeeze(1)
            return (torch.sigmoid(o) if task == "clf" else o).cpu().numpy()
    return f

# Classification: tune learning rate and weight decay on dev AUROC
best = (-1, None)
for lr_ in [1e-3, 1e-2]:
    for wd_ in [1e-4, 1e-3]:
        net = train_nn("clf", lr_, wd_)
        a = roc_auc_score(yc[dv], nn_pred(net, "clf")(Xs[dv]))
        if a > best[0]: best = (a, net)
eval_clf("NN", nn_pred(best[1], "clf"))

# Regression: tune on dev R2
best = (-9, None)
for lr_ in [1e-3, 1e-2]:
    for wd_ in [1e-4, 1e-3]:
        net = train_nn("reg", lr_, wd_)
        a = r2_score(yrs[dv], nn_pred(net, "reg")(Xs[dv]))
        if a > best[0]: best = (a, net)
eval_reg("NN", nn_pred(best[1], "reg"))

print(sorted({k[0] for k in results}))   # expect KMeans, LogReg, NN, RF

['KMeans', 'LogReg', 'NN', 'RF']


## Comparison with the Paper and Reasons for Differences

**What matches.** Random forest is the best classifier in both the paper and my reproduction (test kappa 0.59 vs 0.52, AUROC 0.91 vs 0.71) and also the best regressor (R² 0.476 vs 0.38). K-means regression has near-zero or negative skill in both (R² -0.111 vs 0.037).

**What differs.** My classification scores are higher than the paper's for every model: LogReg (kappa 0.36 vs 0.28, AUROC 0.74 vs 0.60), K-means (0.51 vs 0.24, 0.80 vs 0.63), RF (0.59 vs 0.52, 0.91 vs 0.71) and NN (0.52 vs 0.29, 0.87 vs 0.61). The gap is largest for K-means and the NN. In the paper these two and LogReg are all weak (kappa 0.24 to 0.29), while in my run the NN and K-means come close to the random forest, so the ranking of the weaker models is not reproduced. In regression, my RF has a higher RMSE than the paper (0.75 vs 0.64) but a higher R² (0.476 vs 0.38), and my NN is clearly better (RMSE 0.77 vs 0.90, R² 0.444 vs 0.19). RMSE values are not strictly comparable, because RMSE depends on the spread of the time-to-eruption target, which depends on my eruption dates. Only test-set values are compared with the paper; the paper's train and dev numbers (Figures 4 and 5) were not compared.

**Overfitting.** The full results table at the end of this notebook shows that the random forest memorizes the training set (train kappa 0.999 and AUROC 1.000, against dev kappa 0.59 and AUROC 0.89). The NN also overfits, though less (train kappa 0.64 vs test 0.52). The test scores are what I compare with the paper.

Likely reasons for the differences:

1. **Eruption labels and year leakage.** The paper's eruption periods are not published. In my labels, all 2015 and 2016 earthquakes are "erupting" (one long block), and most other years are 0% erupting. Location, depth and rate features therefore act partly as a proxy for the year, which makes classification much easier than in the paper. This is a likely cause of the AUROC gaps of 0.14 to 0.26, but I did not test it directly (for example with a chronological split, or by dropping the location features).
2. **Random split leakage.** A random split puts neighboring earthquakes, with overlapping 1/7/30-day rate windows, in both train and test, which inflates scores. The paper shares this design, but the degree depends on the labels.
3. **Catalog size.** My catalog has 5,950 earthquakes against the paper's 3,764, from a different date range and magnitude cutoff. The cutoff was chosen on a coarse grid (steps of 0.5), so it could not match the paper's count closely.
4. **Decision threshold.** The paper does not state its threshold. I tuned it on the dev set to maximize kappa, which raises kappa relative to a fixed cutoff. AUROC does not depend on it, and my AUROC is also higher.
5. **Unreported hyperparameters and run-to-run variation.** NN learning rate, weight decay and epochs, and the random forest tree count and depth grid, were tuned by me on dev. Library versions and seeds also differ. NN results also shift slightly between runs (for example test kappa 0.52 vs 0.48) because of GPU nondeterminism, even with seeds set.
6. **Inconsistency in the paper.** The text reports an RF kappa of 0.42 while its table gives 0.52 on test. I compared against the table.

In [9]:
reported = {("LogReg","clf","test"):[0.28,0.60], ("KMeans","clf","test"):[0.24,0.63],
            ("RF","clf","test"):[0.52,0.71],     ("NN","clf","test"):[0.29,0.61]}
rep_reg  = {("KMeans","test"):[1.02,0.037], ("RF","test"):[0.64,0.38], ("NN","test"):[0.90,0.19]}

missing = [n for n in ("results", "reported", "rep_reg") if n not in globals()]
if missing:
    raise NameError(f"Define/re-run the cells that create: {missing}")

clf_rows = [[m, reported[(m,"clf","test")][0], round(results[(m,"clf","test")]["kappa"],2),
                reported[(m,"clf","test")][1], round(results[(m,"clf","test")]["auroc"],2)]
            for m in ["LogReg","KMeans","RF","NN"]
            if (m,"clf","test") in results and (m,"clf","test") in reported]
clf_df = pd.DataFrame(clf_rows, columns=["model","kappa_rep","kappa_yours","auroc_rep","auroc_yours"])
clf_df["d_kappa"] = (clf_df.kappa_yours - clf_df.kappa_rep).round(2)
clf_df["d_auroc"] = (clf_df.auroc_yours - clf_df.auroc_rep).round(2)
print("\nClassification, test set\n", clf_df.to_string(index=False))

reg_rows = [[m, rep_reg[(m,"test")][0], round(results[(m,"reg","test")]["rmse"],2),
                rep_reg[(m,"test")][1], round(results[(m,"reg","test")]["r2"],3)]
            for m in ["KMeans","RF","NN"]
            if (m,"reg","test") in results and (m,"test") in rep_reg]
reg_df = pd.DataFrame(reg_rows, columns=["model","rmse_rep","rmse_yours","r2_rep","r2_yours"])
reg_df["d_rmse"] = (reg_df.rmse_yours - reg_df.rmse_rep).round(2)
reg_df["d_r2"] = (reg_df.r2_yours - reg_df.r2_rep).round(3)
print("\nRegression, test set\n", reg_df.to_string(index=False))


Classification, test set
  model  kappa_rep  kappa_yours  auroc_rep  auroc_yours  d_kappa  d_auroc
LogReg       0.28         0.36       0.60         0.74     0.08     0.14
KMeans       0.24         0.51       0.63         0.80     0.27     0.17
    RF       0.52         0.59       0.71         0.91     0.07     0.20
    NN       0.29         0.52       0.61         0.87     0.23     0.26

Regression, test set
  model  rmse_rep  rmse_yours  r2_rep  r2_yours  d_rmse   d_r2
KMeans      1.02        1.09   0.037    -0.111    0.07 -0.148
    RF      0.64        0.75   0.380     0.476    0.11  0.096
    NN      0.90        0.77   0.190     0.444   -0.13  0.254


## Part 2: Additional Model (RBF Support Vector Machine)

**Why an SVM.** The reference uses logistic regression, K-means, random forest and a neural network, so an SVM is a new model, and it handles both tasks (SVC for classification, SVR for time-to-eruption regression). Our features are already standardized, which RBF kernels need. It gives a smooth non-linear decision boundary, a useful contrast to the axis-aligned splits of the random forest and to the heavily parameterized neural network (about three million weights; the paper says roughly four million, but four hidden layers of 1,000 units on 7 inputs give about 3.0 million parameters).

**Implementation (same protocol as the other models).**
- Same stratified 70/20/10 split, same scaling (training-set statistics only), same metrics (Cohen's kappa and AUROC for classification, RMSE and R² for regression).
- RBF kernel. C in {0.1, 1, 10, 100} and gamma in {"scale", 0.01, 0.1, 1} were tuned on the dev set (AUROC for SVC, R² for SVR).
- The classification score is the SVC decision function. As for the other classifiers, the decision threshold is chosen on the dev set to maximize kappa and then applied to train and test.
- Selected: SVC with C = 100 and gamma = "scale"; SVR with C = 1 and gamma = 1. The SVC value of C and the SVR value of gamma sit at the upper edge of the grid, so slightly larger values were not tested.

In [10]:
from sklearn.svm import SVC, SVR

# --- Classification: tune C, gamma on dev AUROC ---
best = (-1, None)
for C in [0.1, 1, 10, 100]:
    for g in ["scale", 0.01, 0.1, 1]:
        m = SVC(kernel="rbf", C=C, gamma=g).fit(Xs[tr], yc[tr])
        a = roc_auc_score(yc[dv], m.decision_function(Xs[dv]))
        if a > best[0]: best = (a, m, C, g)
svm_c = best[1]
print("SVC best C, gamma:", best[2], best[3], "| dev AUROC:", round(best[0], 3))
eval_clf("SVM", svm_c.decision_function)   # threshold tuned on dev for kappa, same as other models

# --- Regression: tune C, gamma on dev R2 ---
best = (-9, None)
for C in [0.1, 1, 10, 100]:
    for g in ["scale", 0.01, 0.1, 1]:
        m = SVR(kernel="rbf", C=C, gamma=g).fit(Xs[tr], yrs[tr])
        a = r2_score(yrs[dv], m.predict(Xs[dv]))
        if a > best[0]: best = (a, m, C, g)
svm_r = best[1]
print("SVR best C, gamma:", best[2], best[3], "| dev R2:", round(best[0], 3))
eval_reg("SVM", svm_r.predict)



SVC best C, gamma: 100 scale | dev AUROC: 0.796
SVR best C, gamma: 1 1 | dev R2: 0.316


### Test-set comparison with the Part 1 models

In [11]:
# --- Final comparison on the test set (yours + paper where available) ---
models = ["LogReg", "KMeans", "RF", "NN", "SVM"]
clf_tbl = pd.DataFrame([[m,
    reported.get((m,"clf","test"), [None,None])[0], round(results[(m,"clf","test")]["kappa"],2),
    reported.get((m,"clf","test"), [None,None])[1], round(results[(m,"clf","test")]["auroc"],2)]
    for m in models], columns=["model","kappa_paper","kappa_yours","auroc_paper","auroc_yours"])
reg_tbl = pd.DataFrame([[m,
    rep_reg.get((m,"test"), [None,None])[0], round(results[(m,"reg","test")]["rmse"],2),
    rep_reg.get((m,"test"), [None,None])[1], round(results[(m,"reg","test")]["r2"],3)]
    for m in ["KMeans","RF","NN","SVM"]], columns=["model","rmse_paper","rmse_yours","r2_paper","r2_yours"])
print("Classification (test)\n", clf_tbl.to_string(index=False))
print("\nRegression (test)\n", reg_tbl.to_string(index=False))

Classification (test)
  model  kappa_paper  kappa_yours  auroc_paper  auroc_yours
LogReg         0.28         0.36         0.60         0.74
KMeans         0.24         0.51         0.63         0.80
    RF         0.52         0.59         0.71         0.91
    NN         0.29         0.52         0.61         0.87
   SVM          NaN         0.55          NaN         0.82

Regression (test)
  model  rmse_paper  rmse_yours  r2_paper  r2_yours
KMeans        1.02        1.09     0.037    -0.111
    RF        0.64        0.75     0.380     0.476
    NN        0.90        0.77     0.190     0.444
   SVM         NaN        0.80       NaN     0.407


## Part 2 Results and Discussion

**Test-set summary (our runs; the paper has no SVM).**

| Model | Kappa | AUROC | Regression RMSE | Regression R² |
|---|---|---|---|---|
| LogReg | 0.36 | 0.74 | n/a | n/a |
| K-means | 0.51 | 0.80 | 1.09 | -0.111 |
| Random forest | **0.59** | **0.91** | **0.75** | **0.476** |
| Neural network | 0.52 | 0.87 | 0.77 | 0.444 |
| **SVM (new)** | 0.55 | 0.82 | 0.80 | 0.407 |

**Classification.** The SVM is second on kappa (0.55, behind the random forest at 0.59 and just above the neural network at 0.52) and third on AUROC (0.82, behind the random forest at 0.91 and the neural network at 0.87). It clearly beats logistic regression on both metrics and slightly exceeds K-means (kappa 0.55 vs 0.51, AUROC 0.82 vs 0.80).

**Regression.** The SVM reaches R² 0.41 (RMSE 0.80). That is far better than K-means (negative R²) but below the neural network (0.44) and the random forest (0.48), so it ranks third of four.

**Overfitting.** The SVM generalizes more evenly than the tree and network models. Its kappa goes from 0.60 (train) to 0.55 (test), against 0.999 to 0.59 for the random forest and 0.64 to 0.52 for the neural network. Its regression R² goes from 0.49 to 0.41.

**Interpretation.** Overall the SVM performs similarly to the neural network and slightly worse than the random forest. A plausible explanation for the random forest's advantage is that tree splits on location and earthquake rate separate our eruption labels well, including the year-related structure described in the Part 1 discussion (long 2014-2016 eruption block). A kernel model smooths over such sharp regions, which would be consistent with its lower AUROC. I did not test this explanation directly.

**Caveats.** The test set contains only 110 erupting earthquakes, so kappa differences of about 0.03 (SVM vs neural network) are within noise. On the dev set the order is random forest (0.59), neural network (0.50), SVM (0.48) on kappa. Several models also score higher on test than on dev, which again points to a small, noisy test set. The SVM numbers are higher than the paper's best model (random forest, kappa 0.52), but as explained in Part 1 this reflects our different catalog and eruption labels, so the SVM should be compared with the other models in our own runs rather than with the paper's absolute values. The numbers in the tables and text above were copied from the printed output of one run. NN results vary slightly between runs (GPU nondeterminism), so after any re-run these should be checked against the new output.

## Full results for all models (train / dev / test)

In [12]:
# Full results: train / dev / test for every model (shows overfitting, e.g. RF train vs dev)
rows = []
for (m, task, s), v in results.items():
    if task == "clf":
        rows.append([m, "clf", s, round(v["kappa"], 3), round(v["auroc"], 3), None, None, v["cm"]])
    else:
        rows.append([m, "reg", s, None, None, round(v["rmse"], 3), round(v["r2"], 3), None])
full_df = pd.DataFrame(rows, columns=["model", "task", "split", "kappa", "auroc", "rmse", "r2", "confusion_matrix"])
full_df["split"] = pd.Categorical(full_df["split"], ["train", "dev", "test"], ordered=True)
full_df = full_df.sort_values(["task", "model", "split"])
print(full_df.to_string(index=False))

 model task split  kappa  auroc  rmse     r2          confusion_matrix
KMeans  clf train  0.398  0.753   NaN    NaN  [[3355, 38], [532, 240]]
KMeans  clf   dev  0.343  0.733   NaN    NaN    [[956, 13], [162, 59]]
KMeans  clf  test  0.514  0.797   NaN    NaN      [[478, 7], [63, 47]]
LogReg  clf train  0.372  0.717   NaN    NaN [[3280, 113], [517, 255]]
LogReg  clf   dev  0.307  0.690   NaN    NaN    [[934, 35], [160, 61]]
LogReg  clf  test  0.361  0.743   NaN    NaN     [[460, 25], [71, 39]]
    NN  clf train  0.644  0.924   NaN    NaN  [[3330, 63], [326, 446]]
    NN  clf   dev  0.500  0.842   NaN    NaN    [[946, 23], [125, 96]]
    NN  clf  test  0.522  0.869   NaN    NaN     [[461, 24], [53, 57]]
    RF  clf train  0.999  1.000   NaN    NaN     [[3392, 1], [0, 772]]
    RF  clf   dev  0.590  0.892   NaN    NaN    [[899, 70], [76, 145]]
    RF  clf  test  0.585  0.911   NaN    NaN     [[439, 46], [32, 78]]
   SVM  clf train  0.604  0.887   NaN    NaN  [[3368, 25], [384, 388]]
   SVM